## Staging Data Notebook Template v2.0 (Functional Approach)

This template uses a functional programming approach for producing staging data. The notebook is designed to be executed from a main staging notebook with comprehensive error handling and result reporting.

**Key Features:**
- Pure functional design with strict separation of concerns
- Business logic isolated from error handling
- Centralized error handling with context preservation
- Structured result object for parent notebook integration
- Comprehensive logging with correlation tracking
- Idempotent operations with rollback capability

**v2.0 Changes:**
- `create_staging_dataframe()` is now a pure transformation function
- All validation and error handling moved to orchestration layer
- Cleaner separation between business logic and infrastructure concerns

#### Input
- `stage_context_str` (str): Execution context JSON string containing all global and local parameters that drive the domain logic.

#### Output
JSON dict with:
- `status` (str): "success", "error", or "warning"
- `has_error` (bool): Indicates whether a critical error occurred
- `error_details` (dict): Error type, message, and traceback if applicable
- `metrics` (dict): Execution metrics (source_count, target_count, duration, etc.)
- `logs` (str): Captured log output with correlation tracking


In [ ]:
from spark_quick_start.adapters import FabricAdapter
from spark_quick_start.spark_load_orchestration import create_staging_main_function

In [ ]:
from os.path import abspath
import random
import uuid
import json
import sys
import traceback
import time
from datetime import datetime
from typing import Dict, Any, Optional, Tuple, List
from dataclasses import dataclass, asdict

import logging
from logging.config import dictConfig

from delta.tables import DeltaTable
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import udf, when, col, lit, concat_ws, coalesce, min as spark_min
import pyspark.sql.functions as F


#### Input parameters

In [ ]:
stage_context_str = '''
{}
'''

#### Business Logic - Staging DataFrame Creation

**This is the primary function to customize for your specific staging needs.**

This function is a pure transformation function focused solely on dataframe creation.
All error handling, validation, and logging are managed by the calling function.

In [ ]:
from pyspark.shell import spark


def create_staging_dataframe(
    data_storage_lakehouse_path: str,
    staging_context: Dict[str, Any],
    global_context: Dict[str, Any],
    current_nb_name: str
) -> DataFrame:
    """
    Creates the staging DataFrame for TransactionHistory.

    Reads the AdventureWorks TransactionHistory parquet file named by
    staging_context["src_parquet_path"], adds scdDeletedFlag = 'N' and the row key
    scdRowKey = 'AW2022~<stage>~<TransactionID>', and selects the staged columns.

    Args:
        data_storage_lakehouse_path: Base path for Delta tables
        staging_context: This stage's parameters; src_parquet_path is the source file
        global_context: Global parameters
        current_nb_name: Current notebook name for logging

    Returns:
        DataFrame: Staging data ready for upsert
    """

    # ============================================================================
    # DATA SOURCE DICTIONARY
    # ============================================================================

    data_sources = {
        "src":         staging_context.get("src_parquet_path", "")
   }

    # ============================================================================
    # 1. GET SOURCE DATA
    # ============================================================================

    source_df = spark.read.parquet(data_sources["src"])
   
    enriched_df = (source_df
        .withColumn("scdDeletedFlag", F.lit("N"))

    )

    source_with_key_df = (
        enriched_df
            .withColumn("scdRowKey", concat_ws(
                "~",
                F.lit("AW2022"),
                F.lit("B"),
                F.col("TransactionID")
            )
        )
    )

    staging_df = (source_with_key_df
        .select(
            "scdRowKey",
            "scdDeletedFlag",
            "ActualCost",
            "ModifiedDate",
            "ProductID",
            "Quantity",
            "ReferenceOrderID",
            "ReferenceOrderLineID",
            "TransactionDate",
            "TransactionID",
            "TransactionType"
        )
    )
    

    return staging_df

#### Execute and Return Results

Execute the main function and return results to the parent notebook.

In [ ]:
# Create main function using factory
main = create_staging_main_function(create_staging_dataframe)
        
# Execute
result = main(stage_context_str)